# Práctica 1 (v2): TAD Map y tablas hash

## Objetivos

- Comprender cómo se localizan las cubetas mediante una función hash.
- Explorar y comprobar colisiones resueltas por **encadenamiento**.
- Recorrer una tabla hash de lista de listas, sin modificarla.
- Aplicar un mapa al conteo de frecuencias de palabras.
- Opcionalmente, extender un TAD Map mediante herencia.


## Instrucciones

La práctica consta de **3 retos obligatorios (45 minutos aprox.)** y **1 ampliación opcional (15–20 minutos)**. Cada reto contiene un enunciado, una celda para programar y otra de pruebas con `assert`. Ejecuta las celdas en orden.

La clase base ya está implementada; **no la modifiques**. Intenta utilizar los métodos públicos de `Map` cuando sea posible. Los retos que estudian la estructura interna indicarán expresamente cuándo consultar `table` y `_hash`.

Puedes usar IA para aclarar conceptos o depurar, pero procura escribir tú la implementación.


# 📚 Antes de empezar: ¿cómo funciona esta tabla hash?

La implementación de esta práctica utiliza los mismos conceptos fundamentales de una tabla hash con **encadenamiento**:

- **Tabla y buckets:** `table` es una lista de listas. Cada sublista es una cubeta (*bucket*) que puede contener varios pares clave-valor.
- **Función hash:** `_hash(key)` transforma la clave en un índice. El operador módulo garantiza que el índice quede dentro del rango de la tabla.
- **Colisiones:** dos claves diferentes pueden producir el mismo índice. En ese caso, ambas se almacenan en el mismo `bucket`.
- **Encadenamiento:** cada `bucket` puede contener varios pares `[key, value]`, lo que permite resolver las colisiones.
- **`put(key, value)`:** inserta una nueva asociación o actualiza el valor si la clave ya existe.
- **`get(key)`:** busca la clave únicamente en su `bucket` y devuelve el valor o `None`.
- **`remove(key)`:** elimina la asociación y devuelve su valor, o `None` si no existe.

> **⚠️ Importante**
>
> Un `Map` mantiene **claves únicas**. Los valores, en cambio, sí pueden repetirse.

## Comprobación automática

Después de cada reto encontrarás una celda de pruebas basada en `assert`.

Si todas las pruebas se ejecutan correctamente aparecerá:

`Pruebas superadas`

Si alguna falla, revisa tu implementación y vuelve a ejecutar primero la celda de solución y después la celda de pruebas.

# 🧱 Código base: clase `Map`

La siguiente clase implementa el TAD Map mediante una tabla hash de capacidad fija `7` y resolución de colisiones por encadenamiento.

**Ejecuta esta celda antes de comenzar los retos y no la modifiques.**

In [ ]:
class Map:
    """Implementación de un TAD Map usando una Tabla Hash con encadenamiento."""

    def __init__(self):
        self._capacity = 7
        self.table = [[] for _ in range(self._capacity)]
        self.size = 0

    def _hash(self, key):
        """Función hash interna que calcula la posición en la tabla."""
        return abs(hash(key)) % self._capacity

    def put(self, key, value):
        """Añade una asociación o actualiza el valor si la clave ya existe."""
        index = self._hash(key)
        bucket = self.table[index]

        # Si la clave ya existe, se actualiza su valor (Clave Única).
        for item in bucket:
            if item[0] == key:
                item[1] = value
                return

        # Si es una clave nueva, se agrega [key, value].
        bucket.append([key, value])
        self.size += 1

    def get(self, key):
        """Devuelve el valor asociado a la clave, o None si no existe."""
        index = self._hash(key)
        bucket = self.table[index]

        for item in bucket:
            if item[0] == key:
                return item[1]
        return None

    def remove(self, key):
        """Elimina el par clave-valor correspondiente a la clave."""
        index = self._hash(key)
        bucket = self.table[index]

        for i in range(len(bucket)):
            if bucket[i][0] == key:
                self.size -= 1
                return bucket.pop(i)[1]
        return None

    def __len__(self):
        return self.size

    def display(self):
        """Muestra la estructura interna de la tabla."""
        print("\n--- Map Contents ---")
        for i in range(self._capacity):
            print(f"Index {i}: {self.table[i]}")
        print(f"Total elements: {len(self)}")
        print("--------------------\n")

# 🏆 Reto 1. Explorar colisiones

🎯 **Objetivo:** entender qué sucede cuando dos claves comparten una misma cubeta.

🕒 **Tiempo:** 10 minutos · 🟢 **Dificultad:** fácil.

Implementa `find_collision(map_obj)`, que encuentre **dos enteros distintos** entre 0 y 99 cuyo índice `_hash` sea igual. Devuelve `(clave1, clave2, indice)`. No insertes nada dentro de esta función.

Después, las pruebas insertarán ambas claves con `put`, comprobarán que conviven en la misma cubeta, recuperarán sus valores y eliminarán una sin perder la otra.

> 💡 Pista: guarda en un diccionario auxiliar el primer entero que encuentres para cada índice. También puedes comparar pares de enteros directamente.


### 💻 Tu solución

In [ ]:
def find_collision(map_obj: Map) -> tuple:
    """Devuelve (a, b, indice) con a != b y _hash(a) == _hash(b)."""
    # TODO: encuentra una colisión entre enteros de 0 a 99
    pass


### ✅ Comprobación automática

In [ ]:
m1 = Map()
a, b, index = find_collision(m1)
assert isinstance(a, int) and isinstance(b, int) and a != b
assert 0 <= a < 100 and 0 <= b < 100
assert m1._hash(a) == m1._hash(b) == index
assert len(m1) == 0  # La búsqueda de la colisión no inserta elementos.

m1.put(a, "primero")
m1.put(b, "segundo")
assert len(m1) == 2
assert len(m1.table[index]) == 2
assert m1.get(a) == "primero" and m1.get(b) == "segundo"
assert m1.remove(a) == "primero"
assert m1.get(a) is None and m1.get(b) == "segundo"
assert len(m1) == 1 and len(m1.table[index]) == 1
print("Pruebas superadas")


# 🏆 Reto 2. Obtener todas las claves y valores

🎯 **Objetivo:** recorrer todas las cubetas de la tabla hash.

🕒 **Tiempo:** 15 minutos · 🟢 **Dificultad:** fácil.

Implementa `get_keys(map_obj)` y `get_values(map_obj)`. Cada función debe devolver una lista con las claves o valores, respectivamente, recorriendo **todos** los buckets. No modifiques el mapa. Los valores pueden repetirse y no debes asumir un orden concreto.

> 💡 Pista: necesitas un bucle por cubeta y otro por cada par `[key, value]`.


### 💻 Tu solución

In [ ]:
def get_keys(map_obj: Map) -> list:
    """Devuelve una lista con todas las claves contenidas en el mapa."""
    # TODO: implementa esta función
    pass


def get_values(map_obj: Map) -> list:
    """Devuelve una lista con todos los valores contenidos en el mapa."""
    # TODO: implementa esta función
    pass

### ✅ Comprobación automática

In [ ]:
m2 = Map()
m2.put(0, "Ana")
m2.put(7, "Carlos")  # Colisiona con 0: ambos índices son 0.
m2.put(14, "Ana") # Tres elementos en la misma cubeta.
m2.put(1, "Lucía")

keys = get_keys(m2)
values = get_values(m2)
assert sorted(keys) == [0, 1, 7, 14]
assert sorted(values) == ["Ana", "Ana", "Carlos", "Lucía"]
assert len(keys) == len(values) == len(m2) == 4
assert len(m2.table[0]) == 3
assert m2.get(0) == "Ana" and m2.get(7) == "Carlos"
assert m2.get(14) == "Ana" and m2.get(1) == "Lucía"

empty = Map()
assert get_keys(empty) == [] and get_values(empty) == []
print("Pruebas superadas")


# 🏆 Reto 3. Contador de frecuencia de palabras

🎯 **Objetivo:** aplicar `Map` para contar palabras en un texto.

🕒 **Tiempo:** 20 minutos · 🟡 **Dificultad:** intermedia.

Implementa `WordCounter` con un `self.freq_map = Map()` y los métodos:

- `add_text(text)`: convierte a minúsculas, separa con `text.lower().split()` y acumula frecuencias, también entre llamadas sucesivas.
- `get_count(word)`: devuelve la frecuencia de la palabra ignorando mayúsculas o `0` si no existe.
- `most_frequent()`: devuelve `(palabra, frecuencia)` de una palabra con frecuencia máxima, o `(None, 0)` si está vacío. En caso de empate, **cualquiera de las palabras empatadas es válida**.

No elimines signos de puntuación: el tokenizado es exactamente `text.lower().split()`.

> 💡 Pista: consulta con `get()` y actualiza con `put()`. Para hallar el máximo, recorre todas las cubetas.


### 💻 Tu solución

In [ ]:
class WordCounter:
    def __init__(self):
        self.freq_map = Map()

    def add_text(self, text: str):
        # TODO
        pass

    def get_count(self, word: str) -> int:
        # TODO
        pass

    def most_frequent(self) -> tuple:
        # TODO
        pass

### ✅ Comprobación automática

In [ ]:
wc = WordCounter()
assert wc.most_frequent() == (None, 0)
assert wc.get_count("hola") == 0
wc.add_text("Hola mundo hola python mundo hola")
assert wc.get_count("hola") == 3
assert wc.get_count("HOLA") == 3
assert wc.get_count("python") == 1
assert wc.get_count("java") == 0
assert wc.most_frequent() == ("hola", 3)

wc.add_text("MUNDO mundo")
assert wc.get_count("mundo") == 4
assert wc.most_frequent() == ("mundo", 4)
wc.add_text("  ")
assert wc.get_count("mundo") == 4

other = WordCounter()
other.add_text("uno dos")
word, count = other.most_frequent()
assert word in {"uno", "dos"} and count == 1
print("Pruebas superadas")


# ⭐ Ampliación opcional. Extender `Map` con `AdvancedMap`

🕒 **Tiempo:** 15–20 minutos · 🟡 **Dificultad:** intermedia.

Crea una subclase `AdvancedMap(Map)` con dos métodos:

- `clear()`: vacía todas las cubetas y restablece `size` a cero.
- `keys_for_value(target_value)`: devuelve una lista con todas las claves asociadas al valor, o `[]` si no hay coincidencias.

Es una ampliación de **herencia y operaciones sobre el TAD**: no es necesaria para superar los tres retos principales.


### 💻 Tu solución

In [ ]:
class AdvancedMap(Map):
    def clear(self):
        # TODO
        pass

    def keys_for_value(self, target_value) -> list:
        # TODO
        pass


### ✅ Comprobación automática (opcional)

In [ ]:
adv = AdvancedMap()
adv.put("A1", "Sistemas")
adv.put("A2", "Derecho")
adv.put("A3", "Sistemas")
assert sorted(adv.keys_for_value("Sistemas")) == ["A1", "A3"]
assert adv.keys_for_value("Medicina") == []
adv.clear()
assert len(adv) == 0
assert adv.get("A1") is None
assert all(bucket == [] for bucket in adv.table)
adv.put("B1", "Matemáticas")
assert len(adv) == 1 and adv.get("B1") == "Matemáticas"
print("Pruebas superadas")


# 🎉 Final de la práctica

Al completar los tres retos principales habrás comprobado qué es una colisión y cómo se resuelve por encadenamiento, cómo recorrer todos los buckets y cómo usar `Map` para contar frecuencias. Si has hecho la ampliación, también habrás practicado herencia.
